In [1]:
import os
import pickle

base_results_dir = os.path.join('..', 'experiment_one_results')

# New Optimized Structure: 
# results[scenario][model_name][context_size][fold] -> List of nllh values (one per seed)
results = {}

print(f"Scanning for files in: {os.path.abspath(base_results_dir)} ...\n")

for root, dirs, files in os.walk(base_results_dir):
    for file in files:
        if file.endswith('.pkl'):
            file_path = os.path.join(root, file)
            
            try:
                with open(file_path, 'rb') as f:
                    data = pickle.load(f)
                
                config = data['config']
                metrics = data['metrics']
                
                # Extract keys
                scenario = config['scenario']
                model_name = config['model_name']
                fold = config['fold']
                context_size = config['context_size']
                
                # We don't need 'seed' as a key anymore, we just append the value
                nllh = metrics['nllh']
                
                # 1. Ensure keys exist
                if scenario not in results:
                    results[scenario] = {}
                    
                if model_name not in results[scenario]:
                    results[scenario][model_name] = {}
                    
                if context_size not in results[scenario][model_name]:
                    results[scenario][model_name][context_size] = {}
                    
                if fold not in results[scenario][model_name][context_size]:
                    results[scenario][model_name][context_size][fold] = []
                
                # 2. Append the NLLH (aggregating seeds into a list)
                results[scenario][model_name][context_size][fold].append(nllh)

            except Exception as e:
                print(f"Error processing file {file}: {e}")

print("Results dictionary built successfully.")

Scanning for files in: c:\Users\ihagv\Desktop\masters_project\project_code\cluster\experiment_one_results ...

Results dictionary built successfully.


In [2]:
# Example verification (replace keys with actual values you expect)
# val = results['scenario_name']['model_name'][seed_value][fold_value][context_size_value]
# print(f"NLLH: {val}")

# Or simply inspect the keys of the first scenario to ensure it loaded correctly
first_scenario = list(results.keys())[0]
print(f"Loaded Scenarios: {list(results.keys())}")
print(f"Models in '{first_scenario}': {list(results[first_scenario].keys())}")

Loaded Scenarios: ['clasp_factoring', 'lpg-zeno', 'saps-CVVAR', 'spear_qcp', 'spear_swgcp', 'yalsat_qcp', 'yalsat_swgcp']
Models in 'clasp_factoring': ['distnet', 'tabpfn']


In [3]:
import numpy as np

plot_vals = {}

# Iterate over the optimized results structure
for scenario, models in results.items():
    plot_vals[scenario] = {}
    
    for model_name, contexts in models.items():
        plot_vals[scenario][model_name] = {}
        
        for context_size, folds_data in contexts.items():
            # folds_data is now: { fold_id: [nllh_seed1, nllh_seed2, ...] }
            
            fold_averages = []
            
            # 1. Calculate average across seeds for each fold
            for fold, nllh_list in folds_data.items():
                avg_seed_nllh = np.mean(nllh_list)
                fold_averages.append(avg_seed_nllh)
            
            # 2. Calculate Mean and Std across the 10 folds
            # We convert to numpy array first for safety/speed
            fold_averages_np = np.array(fold_averages)
            
            plot_vals[scenario][model_name][context_size] = {
                'mean': np.mean(fold_averages_np),
                'std': np.std(fold_averages_np)
            }

print("plot_vals calculation complete.")
# Verification: Print the first entry found
first_scen = list(plot_vals.keys())[0]
first_mod = list(plot_vals[first_scen].keys())[0]
print(f"Example Data [{first_scen}][{first_mod}]: {plot_vals[first_scen][first_mod]}")

plot_vals calculation complete.
Example Data [clasp_factoring][distnet]: {1024: {'mean': np.float64(-0.15116629928502964), 'std': np.float64(0.01709835301600062)}, 128: {'mean': np.float64(-0.007344355460771829), 'std': np.float64(0.039932076163209276)}, 16: {'mean': np.float64(0.4179231831698882), 'std': np.float64(0.06592235663888167)}, 2048: {'mean': np.float64(-0.16066429788149855), 'std': np.float64(0.021595277572831535)}, 256: {'mean': np.float64(-0.08565779903366993), 'std': np.float64(0.02993071742193679)}, 2: {'mean': np.float64(0.3359414215411467), 'std': np.float64(0.15386386135890748)}, 32: {'mean': np.float64(0.30526369614598037), 'std': np.float64(0.08011757894498031)}, 4096: {'mean': np.float64(-0.16466520358599304), 'std': np.float64(0.02186549862329541)}, 4: {'mean': np.float64(0.3747684654786756), 'std': np.float64(0.08512780015518694)}, 512: {'mean': np.float64(-0.1318217394404116), 'std': np.float64(0.022791233490206113)}, 64: {'mean': np.float64(0.14792484686071222

In [5]:
import plotly.graph_objects as go
import plotly.colors as pc
from plotly.subplots import make_subplots
import numpy as np
from collections import defaultdict
import math

def plot_results_grid(results_dict, min_context_size=0):
    """
    Compiles all scenario plots into a single large canvas with 3 plots per row.
    """
    
    # 1. Setup Grid Dimensions
    scenarios = list(results_dict.keys())
    num_scenarios = len(scenarios)
    cols = 2
    rows = math.ceil(num_scenarios / cols)
    
    fig = make_subplots(
        rows=rows, 
        cols=cols, 
        subplot_titles=[f"{s}" for s in scenarios],
        vertical_spacing=0.08,
        horizontal_spacing=0.05
    )
    
    palette = pc.qualitative.Plotly
    models_in_legend = set()

    # --- 2. Iterate through Scenarios ---
    for idx, (scenario, models) in enumerate(results_dict.items()):
        
        row = (idx // cols) + 1
        col = (idx % cols) + 1
        
        # --- DATA PREP ---
        points_at_x = defaultdict(list)
        
        for model_name, context_data in models.items():
            for context_size, stats in context_data.items():
                if context_size < min_context_size: continue
                points_at_x[context_size].append((model_name, stats['mean']))

        if not points_at_x:
            continue

        # --- RANKING POSITIONS ---
        label_decisions = defaultdict(dict)
        for x_val, data_list in points_at_x.items():
            data_list.sort(key=lambda item: item[1]) 
            count = len(data_list)
            for rank, (m_name, val) in enumerate(data_list):
                if count == 1: pos = "top center"
                elif rank == 0: pos = "bottom center"
                elif rank == count - 1: pos = "top center"
                else: pos = "middle right" if rank % 2 == 0 else "middle left"
                label_decisions[m_name][x_val] = pos

        # --- PLOTTING TRACES ---
        all_x_ticks_in_subplot = set()
        
        for i, (model_name, context_data) in enumerate(models.items()):
            x_vals, means, stds, text_positions = [], [], [], []
            
            for context_size, stats in context_data.items():
                if context_size < min_context_size: continue
                x_vals.append(context_size)
                means.append(stats['mean'])
                stds.append(stats['std'])
                all_x_ticks_in_subplot.add(context_size)
                text_positions.append(label_decisions[model_name].get(context_size, "top center"))
            
            if not x_vals: continue

            # Sort
            sorted_indices = np.argsort(x_vals)
            x_vals = np.array(x_vals)[sorted_indices]
            means = np.array(means)[sorted_indices]
            stds = np.array(stds)[sorted_indices]
            text_positions = np.array(text_positions)[sorted_indices]
            
            color = palette[i % len(palette)]
            
            show_legend = False
            if model_name not in models_in_legend:
                show_legend = True
                models_in_legend.add(model_name)

            # 1. Std Ribbon
            x_poly = np.concatenate([x_vals, x_vals[::-1]])
            y_poly = np.concatenate([means + stds, (means - stds)[::-1]])
            
            fig.add_trace(go.Scatter(
                x=x_poly, y=y_poly,
                fill='toself',
                fillcolor=f"rgba{tuple(int(color.lstrip('#')[i:i+2], 16) for i in (0, 2, 4)) + (0.15,)}",
                line=dict(color='rgba(255,255,255,0)'),
                hoverinfo="skip",
                showlegend=False,
                legendgroup=model_name
            ), row=row, col=col)
            
            # 2. Main Line
            # FIX: We use double curly braces {{ }} for Plotly variables inside f-strings
            hover_template_str = (
                f"<b>{model_name}</b><br>" +
                "NLLH: %{y:.4f}<br>" +
                "Std: ±%{customdata:.4f}<extra></extra>"
            )

            fig.add_trace(go.Scatter(
                x=x_vals, y=means,
                mode='lines+markers+text',
                name=model_name,
                line=dict(color=color, width=3),
                marker=dict(size=6),
                text=[f"<b>{m:.3f}</b>" for m in means],
                textposition=text_positions,
                textfont=dict(size=9, color=color, family="Arial Black"),
                
                # --- FIXED LINE BELOW ---
                hovertemplate=hover_template_str,
                
                customdata=stds,
                showlegend=show_legend,
                legendgroup=model_name
            ), row=row, col=col)

        # Update Axes
        sorted_ticks = sorted(list(all_x_ticks_in_subplot))
        fig.update_xaxes(type="log", tickvals=sorted_ticks, ticktext=sorted_ticks, row=row, col=col)
        fig.update_yaxes(zeroline=False, gridcolor='lightgray', row=row, col=col)

    # --- 3. Final Layout ---
    fig.update_layout(
        title=dict(text=f"Experiment Results Overview (Context >= {min_context_size})", font=dict(size=24)),
        template="plotly_white",
        height=400 * rows, 
        width=1300,        
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    fig.show()

# --- Example Usage ---
plot_results_grid(plot_vals, min_context_size=20)

In [6]:
import pandas as pd
import numpy as np

def generate_scenario_tables(results_dict):
    """
    Converts the results dictionary into Pandas DataFrames for each scenario.
    Returns a dictionary of DataFrames: {scenario_name: dataframe}
    """
    scenario_dfs = {}

    for scenario, models in results_dict.items():
        # 1. Collect all unique context sizes for this scenario to define columns
        all_context_sizes = set()
        for model_data in models.values():
            all_context_sizes.update(model_data.keys())
        
        # Sort context sizes numerically for logical column ordering
        sorted_contexts = sorted(list(all_context_sizes))
        
        # 2. Build the data rows
        rows = []
        model_names = []
        
        for model_name, context_data in models.items():
            model_names.append(model_name)
            row_data = {}
            
            for ctx in sorted_contexts:
                if ctx in context_data:
                    mean = context_data[ctx]['mean']
                    std = context_data[ctx]['std']
                    # Format: "Mean ± Std"
                    row_data[ctx] = f"{mean:.4f} ± {std:.4f}"
                else:
                    # Handle missing data (optional)
                    row_data[ctx] = "-"
            
            rows.append(row_data)
        
        # 3. Create DataFrame
        df = pd.DataFrame(rows, index=model_names)
        
        # Ensure columns are sorted (Pandas might reorder them otherwise)
        df = df[sorted_contexts]
        
        # Add a name to the index for clarity
        df.index.name = "Model"
        
        scenario_dfs[scenario] = df

    return scenario_dfs

# --- Usage ---
tables = generate_scenario_tables(plot_vals)

# Display the tables
for scenario, df in tables.items():
    print(f"\n=== Results for Scenario: {scenario} ===")
    display(df) # Use 'display(df)' in Jupyter, or 'print(df)' in standard script
    
    # Optional: Save to CSV
    # df.to_csv(f"../experiment_one_results/table_{scenario}.csv")


=== Results for Scenario: clasp_factoring ===


,2,4,8,16,32,64,128,256,512,1024,2048,4096,8192
Model,,,,,,,,,,,,,
distnet,0.3359 ± 0.1539,0.3748 ± 0.0851,0.3958 ± 0.0569,0.4179 ± 0.0659,0.3053 ± 0.0801,0.1479 ± 0.0604,-0.0073 ± 0.0399,-0.0857 ± 0.0299,-0.1318 ± 0.0228,-0.1512 ± 0.0171,-0.1607 ± 0.0216,-0.1647 ± 0.0219,-0.1697 ± 0.0222
tabpfn,1.1077 ± 0.7638,0.3660 ± 0.3170,0.0691 ± 0.0672,0.0525 ± 0.1271,-0.0567 ± 0.0517,-0.1228 ± 0.0255,-0.1620 ± 0.0207,-0.1868 ± 0.0200,-0.2004 ± 0.0200,-0.2110 ± 0.0178,-0.2185 ± 0.0159,-0.2200 ± 0.0134,-0.2109 ± 0.0128



=== Results for Scenario: lpg-zeno ===


,2,4,8,16,32,64,128,256,512,1024,2048,4096,8192
Model,,,,,,,,,,,,,
distnet,0.0333 ± 0.2313,-0.0260 ± 0.1550,0.0137 ± 0.0979,-0.0340 ± 0.0877,-0.0269 ± 0.0888,-0.0887 ± 0.0901,-0.2925 ± 0.1000,-0.6158 ± 0.0501,-0.7244 ± 0.0256,-0.7667 ± 0.0123,-0.8029 ± 0.0091,-0.8184 ± 0.0106,-0.8299 ± 0.0088
tabpfn,0.7362 ± 1.1685,-0.1923 ± 0.2793,-0.4824 ± 0.1809,-0.6579 ± 0.0820,-0.7481 ± 0.0572,-0.8444 ± 0.0292,-0.8834 ± 0.0219,-0.9215 ± 0.0152,-0.9451 ± 0.0098,-0.9582 ± 0.0104,-0.9664 ± 0.0095,-0.9702 ± 0.0097,-0.9711 ± 0.0107



=== Results for Scenario: saps-CVVAR ===


...

,2,4,8,16,32,64,128,256,512,1024,2048,4096,8192
Model,,,,,,,,,,,,,
distnet,1.5058 ± 0.7962,1.9438 ± 0.9278,2.3525 ± 1.0459,1.9716 ± 0.4357,1.2588 ± 0.2929,0.6803 ± 0.0934,0.4568 ± 0.0765,0.2925 ± 0.0778,0.0045 ± 0.0529,-0.3345 ± 0.0215,-0.4135 ± 0.0194,-0.4402 ± 0.0132,-0.4514 ± 0.0147
tabpfn,1.2567 ± 0.9110,0.5347 ± 0.2698,0.2197 ± 0.1093,-0.0227 ± 0.0985,-0.1758 ± 0.0512,-0.3249 ± 0.0253,-0.3964 ± 0.0182,-0.4302 ± 0.0164,-0.4557 ± 0.0173,-0.4738 ± 0.0173,-0.4889 ± 0.0162,-0.5006 ± 0.0136,-0.5002 ± 0.0130



=== Results for Scenario: yalsat_qcp ===


,2,4,8,16,32,64,128,256,512,1024,2048,4096,8192
Model,,,,,,,,,,,,,
distnet,1.1052 ± 0.9383,1.1644 ± 1.1869,2.0873 ± 1.6471,1.6477 ± 0.9848,1.2521 ± 0.2451,0.9638 ± 0.1225,0.7765 ± 0.0997,0.3411 ± 0.1430,-0.4146 ± 0.0625,-0.6292 ± 0.0214,-0.6972 ± 0.0140,-0.7295 ± 0.0095,-0.7428 ± 0.0088
tabpfn,1.9396 ± 1.1830,0.1145 ± 0.3399,-0.1735 ± 0.1010,-0.3453 ± 0.0854,-0.5247 ± 0.0429,-0.6252 ± 0.0188,-0.6880 ± 0.0140,-0.7264 ± 0.0133,-0.7520 ± 0.0091,-0.7698 ± 0.0095,-0.7810 ± 0.0091,-0.7901 ± 0.0079,-0.7930 ± 0.0081



=== Results for Scenario: yalsat_swgcp ===


,2,4,8,16,32,64,128,256,512,1024,2048,4096,8192
Model,,,,,,,,,,,,,
distnet,0.7009 ± 0.7135,1.7817 ± 1.6264,2.7903 ± 1.8705,1.7020 ± 0.4286,0.8021 ± 0.3137,0.3391 ± 0.0883,0.0971 ± 0.0697,-0.1754 ± 0.0423,-0.4543 ± 0.0404,-0.7072 ± 0.0152,-0.7663 ± 0.0152,-0.7902 ± 0.0112,-0.7975 ± 0.0120
tabpfn,0.8959 ± 0.7043,-0.0109 ± 0.3160,-0.3519 ± 0.1733,-0.5538 ± 0.0449,-0.6536 ± 0.0301,-0.7102 ± 0.0222,-0.7621 ± 0.0198,-0.7913 ± 0.0155,-0.8059 ± 0.0150,-0.8147 ± 0.0121,-0.8198 ± 0.0116,-0.8220 ± 0.0111,-0.8226 ± 0.0108
